# `swift_too` module

## SwiftCalendar example - checking when a TOO was scheduled

### API version = 1.2, `swifttools` version = 4.0

#### Author: Jamie A. Kennea (Penn State)

This notebook shows how to use `SwiftCalendar` (shorthand `Calendar`) to look up scheduled windows for a Swift TOO request.

In plain terms: this tells you *when* operations planned to run the TOO, and roughly how much time was actually flown in each window (the `AFST` column).

A practical way to start is to get a recent approved TOO, then feed its TOO ID into `Calendar`.

In [ ]:
from swifttools.swift_too import Calendar

### Step 1: choose a specific TOO ID

For this example we will directly fetch calendar information for TOO `24126`.

In [ ]:
too_id = 24126
print(f"Using TOO ID: {too_id}")

In [ ]:
cal = Calendar(too_id=too_id)
cal

### Step 2: confirm request status

This helps explain empty tables if the server has no calendar windows for that ID.

In [ ]:
cal.status

### Step 3: query the calendar entries

In [ ]:
cal

The table above lists planned windows. Typical columns are:

- `Start` and `Stop`: the planned time window
- `XRT Mode` / `UVOT Mode`: instrument settings
- `Exposure (s)`: requested time in that window
- `AFST (s)`: estimated time actually flown during that exact window

### Step 4: inspect one entry in detail

Entry objects use the standard API table renderer, so they display as a parameter/value table in notebooks.

In [ ]:
cal[0] if len(cal) > 0 else "No calendar rows returned for this TOO ID."

### Notes

- Calendar windows are planned operations windows, not guaranteed final science yield.
- A low or zero `AFST` in a window can happen if the timeline changed, constraints intervened, or work was shifted.
- For completed observations and final exposure accounting, use `ObsQuery` against the actual data products.

### Step 5: query by date range

You can query the planning calendar by time window directly.

Below, we build a date-range query from the first calendar row when available, and otherwise use a fallback date range.

In [ ]:
if len(cal) > 0 and cal[0].begin is not None and cal[0].end is not None:
    begin_date = cal[0].begin.date()
    end_date = cal[0].end.date()
else:
    begin_date = "2024-01-01"
    end_date = "2024-01-02"

date_cal = Calendar(begin=begin_date, end=end_date)
date_cal

### Step 6: query by sky position (RA/Dec)

You can also search calendar windows around a sky position.

Below, we reuse the first returned calendar row coordinates when possible, otherwise use a fixed fallback position.

In [ ]:
if len(cal) > 0 and cal[0].ra is not None and cal[0].dec is not None:
    query_ra = float(cal[0].ra)
    query_dec = float(cal[0].dec)
else:
    query_ra = 83.6331
    query_dec = 22.0145

radec_cal = Calendar(ra=query_ra, dec=query_dec, radius=0.2)
radec_cal

### Cross-check via TOORequests

As a final check, you can fetch the same TOO using `TOORequests` and inspect the attached `.calendar` attribute from the returned request entry.

In [ ]:
from swifttools.swift_too import TOORequests

too_req = TOORequests(too_id=too_id, detail=False)
too_req

too_req[0].calendar if len(too_req) > 0 else "No TOORequests rows returned for this TOO ID."